# RAG de hipotecas en Colab
Pasos: 1) instalar, 2) subir el proyecto, 3) indexar con tu pipeline, 4) recuperar, 5) generar con un LLM, 6) citar desde los metadatos.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo → **GPU T4** (solo hace falta si usas el modelo local de la opción B).

In [ ]:
!pip -q install pymupdf sentence-transformers anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.5 MB/s eta 0:00:00


## 1. Subir el proyecto
Crea un `proyecto.zip` con tres carpetas: `src/` (tus .py), `datos/` (los PDFs) y `evaluacion/` (el .jsonl). Súbelo aquí.

In [ ]:
from google.colab import files
files.upload()          # elige proyecto.zip
!unzip -q -o proyecto.zip -d /content
!ls /content/proyecto /content/proyecto/datos

Saving trocear.py to trocear.py
Saving cargar.py to cargar.py
Saving embeddings.py to embeddings.py
Saving evaluar.py to evaluar.py
Saving files.zip to files.zip
Saving indice.py to indice.py
Saving limpiar.py to limpiar.py
Saving modelos.py to modelos.py
Saving proyecto.zip to proyecto.zip
Saving rag_colab.ipynb to rag_colab.ipynb
/content/proyecto:
datos  evaluacion  rag_colab.ipynb  src

/content/proyecto/datos:
EJEMPLO_01_tipos_de_hipoteca.pdf    EJEMPLO_05_amortizacion.pdf
EJEMPLO_02_solicitud.pdf	    EJEMPLO_06_glosario.pdf
EJEMPLO_03_documentacion.pdf	    EJEMPLO_07_producto.pdf
EJEMPLO_04_gastos_y_comisiones.pdf


## 2. Indexar (tu código: PDFs → líneas → limpiar → chunks → embeddings)

In [ ]:
!pip install pymupdf sentence-transformers

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.2 MB/s eta 0:00:00


In [ ]:
import os
import sys
from pathlib import Path

# Your Python files are directly inside /content
SRC_DIR = Path("/content")

os.chdir(SRC_DIR)
sys.path.insert(0, str(SRC_DIR))

from cargar import cargar_pdf
from limpiar import limpiar
from trocear import trocear
from embeddings import cargar_modelo, crear_embeddings
from indice import buscar_exacto

print("All imports successful!")

All imports successful!


In [ ]:
chunks = []

for ruta in sorted(Path("/content/proyecto/datos").glob("*.pdf")):
    lineas = cargar_pdf(ruta)
    lineas = limpiar(lineas)
    chunks += trocear(lineas, ruta.name)

print(len(chunks), "chunks")

modelo = cargar_modelo()
vectores = crear_embeddings(chunks, modelo)

29 chunks


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## 3. Recuperar

Customer asks: "What is Euribor?"


1. **Embedding model**

Converts the question into a vector

2.   **Retriever**

Finds the four most similar articles

3. **Retrieved context**

The four articles are passed to the next stage

4. **LLM (next step)**

Uses the retrieved content to formulate an answer

In [ ]:
def recuperar(pregunta, k=4):
    v = modelo.encode(["query: " + pregunta], normalize_embeddings=True)[0]
    return [(float(sim), chunks[pos]) for sim, pos in buscar_exacto(v, vectores, k)]

for sim, c in recuperar("¿Qué es el Euríbor?"):
    print(f"{sim:.3f}  {c.id}  {c.documento}  p.{c.pagina}")

0.865  HIP-GLO-03  EJEMPLO_06_glosario.pdf  p.1
0.812  HIP-TIP-02  EJEMPLO_01_tipos_de_hipoteca.pdf  p.1
0.772  HIP-GLO-04  EJEMPLO_06_glosario.pdf  p.1
0.771  HIP-AMO-03  EJEMPLO_05_amortizacion.pdf  p.1


## 4. Prompt
El modelo cita con `[ID]` dentro del texto. Las citas finales **no** las escribe el modelo: se construyen en código a partir de los metadatos del chunk, y se descartan los IDs que no estén entre los recuperados.

Customer question

"¿Qué es el Euríbor?"

**1. Retrieval**

recuperar() finds the most relevant articles.

**2. Prompt construction**

construir_prompt() combines the retrieved articles and the question.

**3. System instructions**

SISTEMA defines how the LLM must answer.

**4. LLM response**

A concise Spanish answer grounded in the retrieved articles, with source IDs.

In [ ]:
SISTEMA = """Eres un asistente de preguntas frecuentes sobre hipotecas de un banco.
Reglas:
- Responde SOLO con la información de los fragmentos entregados. No uses conocimiento externo.
- Si los fragmentos no contienen la respuesta, di que no tienes esa información y recomienda contactar con el banco. No adivines.
- Si un valor figura como [A CONFIRMAR POR EL BANCO], di que ese dato debe confirmarse con el banco. Nunca inventes cifras.
- No des asesoramiento financiero personalizado ni prometas aprobación, tipos o importes.
- Tras cada afirmación escribe entre corchetes el ID del fragmento que la respalda, por ejemplo [HIP-TIP-01].
- Responde en español, de forma breve y clara."""


def construir_prompt(pregunta, hits):
    bloques = [f"[{c.id}] ({c.seccion})\n{c.texto}" for _, c in hits]
    return "Fragmentos:\n\n" + "\n\n".join(bloques) + f"\n\nPregunta: {pregunta}"

## 5. Generar

Modelo abierto local** en la GPU T4 (gratis, sin clave; calidad menor, sobre todo con modelos pequeños).

In [ ]:
# ---- modelo abierto local ----
_gen = None
def generar_local(sistema, usuario, nombre="Qwen/Qwen2.5-3B-Instruct"):
    global _gen
    if _gen is None:
        import torch
        from transformers import pipeline
        _gen = pipeline("text-generation", model=nombre,
                        torch_dtype=torch.float16, device_map="auto")
    mensajes = [{"role": "system", "content": sistema},
                {"role": "user", "content": usuario}]
    salida = _gen(mensajes, max_new_tokens=400, do_sample=False)
    return salida[0]["generated_text"][-1]["content"]


generar = generar_local

## 6. Responder con citas trazables

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [ ]:
import torch
from transformers import pipeline

_gen = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
def recuperar(pregunta, k=4):
    v = modelo.encode(
        ["query: " + pregunta],
        normalize_embeddings=True
    )[0]

    resultados = buscar_exacto(v, vectores, k)

    return [
        (float(sim), chunks[pos])
        for sim, pos in resultados
    ]

In [ ]:
for nombre in [
    "modelo",
    "vectores",
    "chunks",
    "buscar_exacto",
    "construir_prompt",
    "SISTEMA",
    "_gen"
]:
    print(nombre, "OK" if nombre in globals() else "MISSING")

modelo MISSING
vectores MISSING
chunks MISSING
buscar_exacto MISSING
construir_prompt MISSING
SISTEMA MISSING
_gen OK


In [ ]:
import re

# 1. Respuesta de fallback cuando no hay información suficiente
SIN_RESPUESTA = (
    "No tengo esa información en los documentos disponibles. "
    "Por favor, contacta con el banco para más detalles."
)


# 2. Función principal para responder preguntas con citas trazables
def responder(pregunta, k=4, sim_minima=0.0):

    # Paso 1: recuperar los fragmentos más relevantes
    hits = recuperar(pregunta, k)

    # Paso 2: comprobar si se han encontrado fragmentos
    if not hits:
        return {
            "respuesta": SIN_RESPUESTA,
            "citas": [],
            "hits": []
        }

    # Paso 3: comprobar la similitud del mejor resultado
    if hits[0][0] < sim_minima:
        return {
            "respuesta": SIN_RESPUESTA,
            "citas": [],
            "hits": hits
        }

    # Paso 4: construir el prompt y generar la respuesta con el LLM
    prompt_usuario = construir_prompt(pregunta, hits)
    texto = generar_local(SISTEMA, prompt_usuario)

    # Paso 5: asociar los IDs de los fragmentos recuperados
    por_id = {c.id: c for _, c in hits}

    # Paso 6: extraer las citas mencionadas por el modelo
    ids_mencionados = re.findall(r"\[([^\]]+)\]", texto)

    # Eliminar duplicados y conservar solo IDs recuperados
    citados = list(dict.fromkeys(
        i for i in ids_mencionados if i in por_id
    ))

    # Paso 7: construir las fuentes trazables
    citas = [
        f"{por_id[i].documento}, {i}, p. {por_id[i].pagina}"
        for i in citados
    ]

    # Paso 8: devolver respuesta, citas y fragmentos recuperados
    return {
        "respuesta": texto,
        "citas": citas,
        "hits": hits
    }


# 3. Probar el sistema con una pregunta sobre hipotecas
pregunta = "¿Puedo cancelar la hipoteca antes de tiempo? ¿Me cobran algo?"

r = responder(pregunta)

print("PREGUNTA:")
print(pregunta)

print("\nRESPUESTA:")
print(r["respuesta"])

print("\nFUENTES:")
if r["citas"]:
    for cita in r["citas"]:
        print("-", cita)
else:
    print("No se han identificado citas válidas entre los fragmentos recuperados.")

print("\nFRAGMENTOS RECUPERADOS:")
for similitud, chunk in r["hits"]:
    print(f"- Similitud: {similitud:.3f} | {chunk.id} | "
          f"{chunk.documento} | p. {chunk.pagina}")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PREGUNTA:
¿Puedo cancelar la hipoteca antes de tiempo? ¿Me cobran algo?

RESPUESTA:
Sí, puedes cancelar la hipoteca antes de tiempo. Dependiendo de cómo lo hagas, podrías tener que pagar intereses adicionales. Consulta con el banco para conocer las condiciones específicas, ya que [HIP-AMO-03] indica que la compensación puede estar limitada por la normativa y se indica en el contrato [A CONFIRMAR POR EL BANCO].

FUENTES:
- EJEMPLO_05_amortizacion.pdf, HIP-AMO-03, p. 1

FRAGMENTOS RECUPERADOS:
- Similitud: 0.882 | HIP-AMO-03 | EJEMPLO_05_amortizacion.pdf | p. 1
- Similitud: 0.849 | HIP-AMO-02 | EJEMPLO_05_amortizacion.pdf | p. 1
- Similitud: 0.847 | HIP-AMO-04 | EJEMPLO_05_amortizacion.pdf | p. 1
- Similitud: 0.840 | HIP-AMO-01 | EJEMPLO_05_amortizacion.pdf | p. 1


## 7. Preguntas fuera de alcance (deben negarse, no inventar)

In [ ]:
for p in ["¿Cuál es el tipo de interés que me van a dar a mí?",
          "¿Qué tiempo hará mañana en Madrid?",
          "¿Cuánto cuesta la comisión de apertura?"]:     # existe, pero el valor es [A CONFIRMAR]
    r = responder(p)
    print("P:", p); print("R:", r["respuesta"]); print("   top sim:", round(r["hits"][0][0], 3), "\n")

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


P: ¿Cuál es el tipo de interés que me van a dar a mí?
R: [HIP-TIP-02] [HIP-GLO-03] [HIP-TIP-01] · El tipo de interés que te darán dependerá del tipo de hipoteca que elijas. Si eliges una hipoteca a tipo fijo, el interés será constante durante todo el préstamo. Si optas por una a tipo variable, el interés se ajustará según un índice de referencia como el Euríbor, con un diferencial fijo pactado en el contrato.
   top sim: 0.85 



[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


P: ¿Qué tiempo hará mañana en Madrid?
R: Lo siento, pero no tengo esa información. Para obtener el pronóstico del tiempo, deberías consultar una fuente confiable de meteorología como el sitio web del Servicio Meteorológico Nacional o aplicaciones móviles especializadas. [NO HIPÓNIMO]
   top sim: 0.742 

P: ¿Cuánto cuesta la comisión de apertura?
R: [HIP-GAS-04] [A CONFIRMAR POR EL BANCO].
   top sim: 0.862 



## 8. Evaluación
**Retrieval** (el que ya mides) y **extremo a extremo**: ¿el artículo esperado aparece en las citas finales? Para calibrar `sim_minima`, compara la similitud top-1 de preguntas válidas frente a las de fuera de alcance.

In [ ]:
import json

# Load the evaluation cases using the absolute path
ruta_casos = "/content/proyecto/evaluacion/casos_desarrollo.jsonl"

with open(ruta_casos, encoding="utf-8") as f:
    casos = [json.loads(line) for line in f if line.strip()]

print("Casos cargados:", len(casos))

# Start with a small sample: each case requires one LLM call
N = min(10, len(casos))
ok_retrieval = 0
ok_cita = 0

for caso in casos[:N]:
    esperado = caso["expected_article_ids"][0]

    r = responder(caso["question"])

    # Check whether the expected article was retrieved
    ok_retrieval += any(
        esperado == c.id for _, c in r["hits"]
    )

    # Check whether the expected article was cited in the answer
    ok_cita += any(
        esperado in cita for cita in r["citas"]
    )

print(f"Retrieval@4: {ok_retrieval}/{N}")
print(f"Citado en la respuesta: {ok_cita}/{N}")

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Casos cargados: 32


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Retrieval@4: 10/10
Citado en la respuesta: 7/10


Interpretation:

**1. Your evaluation results**

Retrieval@4
100%
10 out of 10 questions

Cited in answer
70%
7 out of 10 questions



What this tells you: Your retriever found the expected article for every tested question, but your system identified a matching citation in only seven generated answers.

**2. What each metric means**
Retrieval@4 = 10/10

For all ten questions, the expected article appeared among the four retrieved chunks. This is an excellent result for this small test sample and suggests that your embedding and retrieval pipeline is working well.

Cited in the answer = 7/10

In seven cases, the expected article ID appeared in the citations returned by your code. In the other three cases, one of the following may have happened:
- Qwen answered without including the expected article ID.
- Qwen cited a different chunk.
- The answer did not follow the citation format expected by your code.
This does not necessarily mean the answers themselves were incorrect. It means the expected source was not detected in the final citations.